# Módulo Deep Learning
## Actividad 2: Reinforcement Learning: **Frozen lake problem**

- Luis Coronel Hidalgo
- Juan Pablo Feijoo
- Jose Cantos
- Pedro Pitarch Oltra

# Enunciado: Actividad Reinforcement Learning

Resolver el problema del Frozen lake de OpenAI Gym. Documentación: https://www.gymlibrary.dev/environments/toy_text/frozen_lake/

## Objetivos
- Conseguir movermos aleatoriamente hasta cumplir el objetivo
- Conseguir que el agente aprenda con Q-learning
- (Opcional) Probar con otros hiperparámetros
- (Opcional) Modificar la recompensa

## Consideraciones
- No hay penalizaciones
- Si el agente cae en un "hole", entonces done = True y se queda atascado sin poder salir (al igual que ocurre cuando llega al "goal")

## Normas a seguir

- Se debe entregar un **ÚNICO GOOGLE COLAB notebook** (archivo .ipynb) que incluya las instrucciones presentes y su **EJECUCIÓN!!!**.
- Poner el nombre del grupo en el nombre del archivo y el nombre de todos los integrantes del grupo al inicio del notebook.

## Criterio de evaluación

- Seguimiento de las normas establecidas en la actividad.
- Corrección en el uso de algoritmos, modelos y formas idiomáticas en Python.
- El código debe poder ejecutarse sin modificación alguna en Google Colaboratory. -->

Action Space

The agent takes a 1-element vector for actions. The action space is (dir), where dir decides direction to move in which can be:

    0: LEFT

    1: DOWN

    2: RIGHT

    3: UP

## **Instalamos librerías**

In [ ]:
import numpy as np
import time
from IPython import display as IPython_display
import random
from tqdm import trange

import gymnasium as gym # Importado gymnasium como nueva versió de gym, ya que usamos VS Code en lugar de colab

from gym import envs
all_envs = envs.registry.values()
envs_ids = [env_spec.id for env_spec in all_envs]
print(sorted(envs_ids))

# **1. Definición del entorno**

Reward schedule:

    Reach goal(G): +1

    Reach hole(H): 0

    Reach frozen(F): 0

In [ ]:
env = gym.make(
    'FrozenLake-v1',
    desc=None,
    map_name="4x4",
    is_slippery=False,
    render_mode="ansi"
    )

In [ ]:
obs, info = env.reset()
done = False

In [ ]:
print(env.render())

# 2. Definición de hiperparámetros

Definimos el total de estados y acciones del entorno en variables independientes para que, en caso de haber un entorno aleatorio, se pueda trabajar.

In [ ]:
n_states = env.observation_space.n
n_actions = env.action_space.n

print(f"Total de estados: {n_states}")
print(f"Total de acciones: {n_actions}")

In [ ]:
Q = np.zeros((n_states, n_actions))

In [ ]:
alpha = 0.1
gamma = 0.99
epsilon = 0.1
epsilon_min = 0.01
epsilon_decay = 0.99
episodes = 2000
max_steps = 1000

In [ ]:
total_rewards = []

# 3. Entrenamiento

In [ ]:
for episode in trange(episodes, desc="Entrenando"):
    state, _ = env.reset()
    done = False
    episode_reward = 0

    for step in range(max_steps):
        # Política ε-greedy
        if random.uniform(0, 1) < epsilon:
            action = env.action_space.sample()  # Acción aleatoria
        else:
            action = np.argmax(Q[state, :])     # Mejor acción según Q

        new_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # Actualización Q-learning
        Q[state, action] = Q[state, action] + alpha * (
            reward + gamma * np.max(Q[new_state, :]) - Q[state, action]
        )

        state = new_state
        episode_reward += reward

        if done:
            break
    
    total_rewards.append(episode_reward)

    # Decaimiento de epsilon
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

    if episode % 100 == 0 and episode != 0:
        media = np.mean(total_rewards[-100:])
        print(f"\n🎯 Recompensa media últimos 100 episodios: {media:.2f}")

print("¡Entrenamiento completado!")
print(Q)